# Data Cleaning Case Study — The Toy Shop's Messy Order Export 🧹🧸

Same toy shop as before — but this time, imagine the shop's old, ancient computer system just spat out an export of all its orders. Different staff typed things in differently over the years, so the same information got written in a dozen different ways.

**Our job:** turn this mess into something a computer (or a human) can actually trust and calculate with — and build it as *reusable functions*, not a pile of one-off code, so the next messy export takes 30 seconds to clean instead of an hour.

## 0. Why does "clean data" even matter? (the toy box analogy)

Imagine your toy box has 100 toys in it, but:
- Some toy price tags say "₹500", some say "500 rupees", some just say "500"
- Some are labeled "Car" and some "car" and some "CAR " (with a trailing space)
- A few toys got put in twice by accident
- Some tags have no price at all

If someone asks *"what's the average toy price?"*, you literally **cannot answer correctly** until every tag says the same kind of thing. Cleaning isn't busywork — it's the step that makes every later question answerable at all.

In [1]:
import pandas as pd
from cleaning import (
    standardize_category, parse_price, parse_weight_with_flag,
    parse_date, standardize_bool, clean_orders
)

raw = pd.read_csv('data/raw/messy_orders.csv')
raw.shape

(154, 7)

In [2]:
raw.head(10)

,order_id,customer_name,category,price,weight,order_date,express_shipping
0,1016,Vivaan Joshi,Art Supplies,651.16,1641,2025-02-01,No
1,1046,Arjun Reddy,art supplies,478.0,0.16kg,2025-01-10,True
2,1076,Rohan Mehta,Games,Rs.804.26,1160g,08 May 2025,1
3,1122,Mira Kulkarni,toys,"1,474.40",2.87kg,2025-03-26,No
4,1040,Diya Nair,Art Supplies,637.47 INR,264,2025-02-27,No
5,1010,Aarav Sharma,TOYS,"1,005.63",2.16kg,27/06/2025,0
6,1128,Kabir Singh,Games,1452.31 INR,8,2025-01-18,Y
7,1088,Aarav Sharma,toys,650.04 INR,464,2025-02-10,no
8,1115,Sara Khan,art supplies,Rs.1260.69,894,22 Jan 2025,True
9,1043,Aarav Sharma,Games,223.08 INR,2541g,24/04/2025,yes


## 1. Spotting the mess — look before you clean

Before writing any cleaning code, it's worth actually *looking* at what's broken. This is like sorting toys into "obviously fine" and "needs fixing" piles before you decide how to fix each pile.

In [3]:
print("Unique category spellings:", sorted(raw['category'].unique()))

Unique category spellings: [' Games', ' Toys', 'Art  Supplies', 'Art Supplies', 'ArtSupplies', 'BOOKS', 'Book ', 'Books', 'Games', 'TOYS', 'Toy', 'Toys', 'art supplies', 'books', 'games', 'toys']


In [4]:
print("A sample of price formats:")
print(raw['price'].sample(8, random_state=1).tolist())

A sample of price formats:
['Rs.951.59', '339.67', '188.34', '682.71', '775.2', 'Rs.862.01', '843.67', 'Rs.574.1']


In [5]:
print("A sample of weight formats:")
print(raw['weight'].sample(8, random_state=1).tolist())

A sample of weight formats:
['629g', '1621g', '1958', '0.98kg', '2.73kg', '2.66kg', '1023g', '0.99kg']


In [6]:
print("A sample of date formats:")
print(raw['order_date'].sample(8, random_state=1).tolist())

A sample of date formats:
['31/08/2025', '2025-09-09', '03/10/2025', '04 May 2025', '26/06/2025', '02-06-2025', '23/03/2025', '07/07/2025']


In [7]:
print("Unique 'express shipping' spellings:", sorted(raw['express_shipping'].astype(str).unique()))

Unique 'express shipping' spellings: ['0', '1', 'FALSE', 'False', 'N', 'No', 'TRUE', 'True', 'Y', 'Yes', 'no', 'yes']


We can already see 4 different problems: inconsistent category labels, prices written in multiple styles, weights in different units (or no unit at all), dates in 4 different formats, and "yes/no" written a dozen ways. Let's fix each one — as a **function**, not a one-off line of code.

## 2. Categories — same label, different handwriting

**Analogy:** imagine 5 different shop staff labeled the same drawer over the years — "toys", "Toys", "TOYS", " Toys" — it's the same drawer, just written differently by different hands. We need one standard spelling so we can group them together later.

In [8]:
examples = ["toys", " Toys ", "TOYS", "Art  Supplies", "ArtSupplies"]
for e in examples:
    print(f"{e!r:20} -> {standardize_category(e)!r}")

'toys'               -> 'Toys'
' Toys '             -> 'Toys'
'TOYS'               -> 'Toys'
'Art  Supplies'      -> 'Art Supplies'
'ArtSupplies'        -> 'Art Supplies'


**A real bug I hit while building this, left in on purpose:** my first version of `standardize_category` only fixed case and spacing. It missed something — a row that said `"Toy"` (singular) stayed `"Toy"` instead of becoming `"Toys"`, because title-casing a *different word* doesn't magically turn it into the *right* word. Same for `"Book"` vs `"Books"`.

**Why this matters:** case/whitespace cleanup is something a function can safely do on its own. But deciding "Toy" and "Toys" mean the same thing requires a human to actually look at the data and confirm it — a function should never *guess* that two different-looking words mean the same category, because sometimes they genuinely don't. So the fix is an explicit, human-reviewed synonym list (`KNOWN_CATEGORY_SYNONYMS` in `cleaning.py`), not a cleverer auto-detection rule. This is why `pytest` output on the next run matters more than eyeballing one printed example — the test that catches this (`test_standardize_category_maps_known_singular_typos`) is what actually caught the bug, not staring at the notebook.

In [9]:
print(standardize_category("Toy"), "  <- now correctly maps to Toys")
print(standardize_category("book"), " <- now correctly maps to Books")

Toys   <- now correctly maps to Toys
Books  <- now correctly maps to Books


## 3. Prices — one ruler, not four different ones

**Analogy:** imagine 4 friends telling you a toy's price: one says "500", another "Rs.500", another "500 INR", another "1,500.00" (with a comma). They all mean a *number*, just dressed differently. Before you can add them up, you need to strip off the costume and get the plain number underneath.

In [10]:
examples = ["651.16", "Rs.804.26", "637.47 INR", "1,474.40", "", None]
for e in examples:
    print(f"{e!r:20} -> {parse_price(e)!r}")

'651.16'             -> 651.16
'Rs.804.26'          -> 804.26
'637.47 INR'         -> 637.47
'1,474.40'           -> 1474.4
''                   -> None
None                 -> None


## 4. Weights — different rulers entirely (kg vs. g)

**Analogy:** imagine one friend measures a toy's weight with a kitchen scale in *grams*, another uses a luggage scale in *kilograms*. "500" from one friend and "500" from the other mean very different things! We convert everything to one common unit (grams) so they can be compared and added together honestly.

**The tricky part:** some entries in our export are just a bare number with **no unit at all** — like "500". Is that 500 grams, or 500 kilograms (500,000 grams)?! We genuinely can't know for sure. Rather than silently guessing and hiding that guess, our function **flags** these rows so a human can double-check them later — pretending we're sure when we're not would be worse than admitting the uncertainty.

In [11]:
examples = ["1160g", "2.87kg", "500"]
for e in examples:
    value, was_ambiguous = parse_weight_with_flag(e)
    print(f"{e!r:10} -> {value} grams   (unit had to be guessed: {was_ambiguous})")

'1160g'    -> 1160.0 grams   (unit had to be guessed: False)
'2.87kg'   -> 2870.0 grams   (unit had to be guessed: False)
'500'      -> 500.0 grams   (unit had to be guessed: True)


## 5. Dates — the same day, written 4 different ways

**Analogy:** if one friend writes today's date as "11/09/2026" and another writes "09-11-2026", do they mean the same day? In India, 11/09 means 9th November. In the US, 11/09 means 11th September. This is a genuinely common real-world data bug — we handle it here by knowing exactly which formats this specific export uses, and converting all of them to one unambiguous standard (`YYYY-MM-DD`, the ISO format that can't be misread).

In [12]:
examples = ["27/06/2025", "2025-06-27", "06-27-2025", "27 Jun 2025"]
for e in examples:
    print(f"{e!r:15} -> {parse_date(e)}")

'27/06/2025'    -> 2025-06-27 00:00:00
'2025-06-27'    -> 2025-06-27 00:00:00
'06-27-2025'    -> 2025-06-27 00:00:00
'27 Jun 2025'   -> 2025-06-27 00:00:00


## 6. Yes/No fields — the same answer, said in different languages

**Analogy:** "is express shipping on?" got answered as Yes, Y, 1, True, yes, TRUE — all across different rows. These all mean the same thing, they're just said in different "dialects." We translate all of them into one real `True`/`False` that Python (and pandas) can actually use in a filter or a count.

In [13]:
examples = ["Yes","No","Y","N","yes","1","0","True","FALSE","maybe"]
for e in examples:
    print(f"{e!r:8} -> {standardize_bool(e)}")

'Yes'    -> True
'No'     -> False
'Y'      -> True
'N'      -> False
'yes'    -> True
'1'      -> True
'0'      -> False
'True'   -> True
'FALSE'  -> False
'maybe'  -> None


Notice `"maybe"` returns `None` rather than a guess — an unrecognized value should be flagged as unknown, not silently forced into True or False.

## 7. Duplicates — the same toy, counted twice

**Analogy:** if the same toy got put in the box twice by accident, counting the box gives you a wrong (inflated) total. An *exact* duplicate row — every single column identical — is almost certainly a double-log, not two separate real events, so it's safe to drop.

In [14]:
print(f"Exact duplicate rows in the raw export: {raw.duplicated().sum()}")

Exact duplicate rows in the raw export: 8


## 8. Putting it all together — the full pipeline, in one call

This is the entire point of writing these as *functions* instead of one-off notebook code: now cleaning this (or any future) export is a single function call, not re-writing logic from scratch.

In [15]:
clean = clean_orders(raw)
print(f"Raw rows: {len(raw)}  →  Clean rows: {len(clean)} (duplicates removed)")
clean.head(10)

Raw rows: 154  →  Clean rows: 146 (duplicates removed)


,order_id,customer_name,category,price,order_date,express_shipping,weight_grams,weight_unit_was_ambiguous
0,1016,Vivaan Joshi,Art Supplies,651.16,2025-02-01,False,1641.0,True
1,1046,Arjun Reddy,Art Supplies,478.00,2025-01-10,True,160.0,False
2,1076,Rohan Mehta,Games,804.26,2025-05-08,True,1160.0,False
3,1122,Mira Kulkarni,Toys,1474.40,2025-03-26,False,2870.0,False
4,1040,Diya Nair,Art Supplies,637.47,2025-02-27,False,264.0,True
5,1010,Aarav Sharma,Toys,1005.63,2025-06-27,False,2160.0,False
6,1128,Kabir Singh,Games,1452.31,2025-01-18,True,8.0,True
7,1088,Aarav Sharma,Toys,650.04,2025-02-10,False,464.0,True
8,1115,Sara Khan,Art Supplies,1260.69,2025-01-22,True,894.0,True
9,1043,Aarav Sharma,Games,223.08,2025-04-24,True,2541.0,False


## 9. Checking our own work — did the cleaning actually work?

**Analogy:** after cleaning your room, you'd actually look around to check it's clean, not just assume it is because you did some tidying. Same here — we verify each column looks right before trusting it.

In [16]:
print("Category values after cleaning:", sorted(clean['category'].dropna().unique()))

Category values after cleaning: ['Art Supplies', 'Books', 'Games', 'Toys']


In [17]:
print(f"Rows with missing price after cleaning: {clean['price'].isna().sum()}")
print(f"Rows with missing weight after cleaning: {clean['weight_grams'].isna().sum()}")
print(f"Rows with ambiguous weight units (flagged, not guessed silently): {clean['weight_unit_was_ambiguous'].sum()}")
print(f"Rows with unparseable dates: {clean['order_date'].isna().sum()}")

Rows with missing price after cleaning: 4
Rows with missing weight after cleaning: 3
Rows with ambiguous weight units (flagged, not guessed silently): 48
Rows with unparseable dates: 0


In [18]:
clean.dtypes

order_id                              int64
customer_name                        object
category                             object
price                               float64
order_date                   datetime64[ns]
express_shipping                       bool
weight_grams                        float64
weight_unit_was_ambiguous              bool
dtype: object

## 10. Save the clean version

The raw export stays untouched in `data/raw/` (never overwrite your original data — you might need to re-check something against it later). The cleaned result goes in `data/processed/`.

In [19]:
clean.to_csv('data/processed/clean_orders.csv', index=False)
print("Saved data/processed/clean_orders.csv")

Saved data/processed/clean_orders.csv


## 11. Why this is tested, not just eyeballed

Every function in `cleaning.py` has unit tests in `tests/test_cleaning.py` — for example, a test that specifically checks a bare number like `"500"` gets *flagged* as ambiguous rather than silently assumed to be grams. Run them yourself:

```bash
pytest tests/ -v
```

**Why this matters:** eyeballing a notebook's output can miss a broken edge case. A test suite catches it automatically, every single time you touch this code again — which is the actual difference between "I wrote a cleaning script" and "I built a cleaning pipeline I can trust."

## Summary — what "clean" meant here, concretely

| Problem | What we did | Why not just drop it |
|---|---|---|
| Inconsistent category casing/spacing | Standardized to one Title Case form | Losing category info would break every future group-by |
| Mixed price formats | Stripped symbols/text, parsed to float | Prices are needed for every revenue question |
| Mixed weight units (and some missing entirely) | Converted to grams, **flagged** bare numbers as ambiguous | Guessing silently would produce confidently wrong numbers |
| 4 different date formats | Parsed all to one ISO standard | Needed for any time-based analysis |
| Messy Yes/No spellings | Standardized to real `True`/`False` | Needed to filter/count express orders correctly |
| Exact duplicate rows | Dropped | They're double-logged, not real repeat events |
